# EDA 심화 2 · 직접 풀어보는 연습문제 정답지

이 노트북은 `EDA_2_Healthcare_Heart_Disease_Practice.ipynb`의 모든 연습문제에 대한 **표준 모범 코드와 결과 해석**을 제공합니다.


In [ ]:
# [실습 환경 준비]
# 이 셀은 심장 질환 환자 20명의 대표 임상 데이터를 생성합니다. 셀을 먼저 실행해주세요.
import pandas as pd
import numpy as np

hd_df = pd.DataFrame({
    "patient_id": range(1, 21),
    "age": [63, 67, 67, 37, 41, 56, 62, 57, 63, 53, 57, 56, 56, 44, 52, 57, 48, 63, 54, 48],
    "sex": [1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1], # 1: 남, 0: 여
    "cp":  [3, 0, 0, 2, 1, 1, 0, 0, 0, 0, 0, 1, 1, 2, 2, 0, 2, 0, 2, 0], # 흉통 유형 (0~3)
    "trestbps": [145, 160, 120, 130, 130, 120, 140, 120, 140, 140, 140, 140, 130, 120, 172, 150, 110, 150, 160, 122], # 혈압
    "chol": [233, 286, 229, 250, 204, 236, 268, 354, 187, 203, 192, 294, 256, 226, 199, 274, 229, 270, 201, 222], # 콜레스테롤
    "thalach": [150, 108, 129, 187, 172, 178, 160, 163, 144, 155, 148, 153, 142, 179, 162, 112, 168, 111, 163, 186], # 최대 심박수
    "target": [1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1] # 1: 질환 유, 0: 정상
})


## 문제 1. 심장 질환 환자 데이터셋의 기초 통계 및 타겟 분포 확인 [의료 통계]

`hd_df` 환자 데이터를 바탕으로:
1. 데이터셋의 행/열 크기(`shape`)를 출력하세요.
2. 진단 결과(`target`)의 빈도수(`count`)와 백분율 비율(`%`)을 계산하여 클래스 균형도를 확인하세요.
3. 전체 환자의 평균 연령(`age`)과 연령대 최소/최댓값을 구하세요.


In [ ]:
# 문제 1 풀이
print("1. 데이터셋 크기:", hd_df.shape)

# 2. 타겟 분포
target_counts = hd_df["target"].value_counts()
target_pcts = hd_df["target"].value_counts(normalize=True) * 100
target_dist = pd.DataFrame({"환자 수": target_counts, "비율(%)": target_pcts})
print("\n2. 심장 질환 진단 결과 분포:")
print(target_dist)

# 3. 연령 통계
print(f"\n3. 환자 평균 연령: {hd_df['age'].mean():.1f}세 (범위: {hd_df['age'].min()}세 ~ {hd_df['age'].max()}세)")


**결과 해석 / 확인한 점:**  
1. **타겟 분포의 적정성:** 정상군(0) 9명(45.0%), 질환군(1) 11명(55.0%)으로 대략 1:1에 근접한 균형 있는 표본 구성을 보입니다.
2. **환자 코호트 연령대:** 평균 54.8세이며 최소 37세부터 최고 67세까지 중장년층 환자 집단으로 구성되어 심혈관 위험 인자 분석에 적합합니다.


## 문제 2. 성별(sex)에 따른 심장 질환 발병률 비교 교차표 [역학 분석]

심혈관 질환은 성별에 따라 발병률과 증상 양상이 크게 다릅니다.
1. `sex`(1: 남성, 0: 여성)와 `target`(질환 여부) 간의 교차표(`pd.crosstab`)를 작성하세요 (`margins=True` 적용).
2. 성별 내에서의 질환 발병률(남성 중 발병률 vs 여성 중 발병률, %)을 구하세요 (`normalize='index'`).


In [ ]:
# 문제 2 풀이
# 1. 교차표 작성
ct_sex = pd.crosstab(hd_df["sex"], hd_df["target"], margins=True)
ct_sex.index = ["여성 (0)", "남성 (1)", "전체 합계"]
ct_sex.columns = ["정상 (0)", "질환 (1)", "전체 합계"]
print("=== 성별 x 심장 질환 진단 교차표 (빈도수) ===")
print(ct_sex)

# 2. 성별 내 비율
ct_rate = pd.crosstab(hd_df["sex"], hd_df["target"], normalize="index") * 100
print("\n=== 성별 내 발병률 (Row %) ===")
print(f"여성 발병률: {ct_rate.loc[0, 1]:.1f}%")
print(f"남성 발병률: {ct_rate.loc[1, 1]:.1f}%")


**결과 해석 / 확인한 점:**  
1. **성별 간 발병 위험 격차:** 여성 5명 중 3명(60.0%), 남성 15명 중 8명(53.3%)이 질환을 보유하고 있어 두 성별 모두에서 절반 이상의 높은 유병률을 보입니다.
2. **표본 수의 불균형 인지:** 남성 표본(15명)이 여성(5명)보다 3배 많으므로 역학적 결론 도출 시 성별 가중치를 고려해야 함을 시사합니다.


## 문제 3. 흉통 유형(cp)에 따른 질환 유병률과 위험도 순위 [증상 탐색]

흉통(`cp`)은 0(전형적 협심증), 1(비전형적), 2(비협심증 통증), 3(무증상)으로 분류됩니다.
`groupby('cp')`를 사용하여:
1. 각 흉통 유형별 환자 수와 질환자 수, 그리고 발병률(`target`의 평균 * 100)을 구하세요.
2. 흉통 유형 중 질환 발병률이 가장 높은 유형을 도출하세요.


In [ ]:
# 문제 3 풀이
cp_analysis = hd_df.groupby("cp")["target"].agg(
    전체환자수="count",
    질환자수="sum",
    발병률=lambda x: round(x.mean() * 100, 1)
).reset_index()

cp_labels = {0: "0 (전형적)", 1: "1 (비전형적)", 2: "2 (비협심증)", 3: "3 (무증상)"}
cp_analysis["흉통명칭"] = cp_analysis["cp"].map(cp_labels)

print("=== 흉통 유형별 심장 질환 유병률 ===")
print(cp_analysis[["cp", "흉통명칭", "전체환자수", "질환자수", "발병률"]].to_string(index=False))


**결과 해석 / 확인한 점:**  
1. **의학적 반직관적 패턴(Counter-intuitive Insight):** 전형적 흉통(0) 환자는 발병률이 20.0%(10명 중 2명)에 불과한 반면, 비전형적(1)과 비협심증(2) 및 무증상(3) 통증 환자의 발병률은 80~100%에 달했습니다.
2. **임상 진단 주의점:** 흉통이 비전형적이거나 뚜렷하지 않더라도 심장 질환 위험이 매우 높을 수 있으므로 정밀 검진이 필수적임을 시사합니다.


## 문제 4. 정상군 vs 질환군 간 수치형 임상 지표 평균 비교 [임상 지표]

질환 여부(`target`)를 기준으로 정상군(0)과 질환군(1)의 핵심 수치 지표 평균 차이를 비교하세요:
- 분석 대상: 연령(`age`), 안정 시 혈압(`trestbps`), 콜레스테롤(`chol`), 최대 심박수(`thalach`)
1. 두 그룹별 평균값을 데이터프레임으로 집계하세요.
2. 정상군 대비 질환군에서 평균 수치가 더 높은 지표와 더 낮은 지표를 구분하세요.


In [ ]:
# 문제 4 풀이
clinical_metrics = ["age", "trestbps", "chol", "thalach"]
group_means = hd_df.groupby("target")[clinical_metrics].mean().round(1)
group_means.index = ["정상군 (0)", "질환군 (1)"]

print("=== 정상군 vs 질환군 임상 지표 평균치 비교 ===")
print(group_means.T)


**결과 해석 / 확인한 점:**  
1. **심박수의 확연한 차이:** 질환군의 최대 심박수(`thalach`) 평균은 162.7 bpm으로 정상군(131.7 bpm)에 비해 무려 31.0 bpm이나 높게 나타났습니다.
2. **연령 지표:** 오히려 질환군의 평균 연령(51.6세)이 정상군(58.7세)보다 젊은 편으로 나타나, 본 표본에서는 연령보다 심박수 및 흉통 양상이 질환 여부에 더 결정적임을 파악할 수 있습니다.


## 문제 5. 상관계수 행렬을 통한 심장 질환 핵심 상관 피처 Top 3 도출 [상관 분석]

전체 수치형 변수들과 `target` 변수 간의 피어슨 상관계수를 산출하세요:
1. `hd_df.corr()['target']`을 구하세요 (`patient_id` 제외).
2. `target`과의 절대값 상관계수(`abs()`)가 가장 높은 순서대로 정렬하여 Top 3 피처를 선정하세요.


In [ ]:
# 문제 5 풀이
corr_matrix = hd_df.drop(columns=["patient_id"]).corr()
target_corr = corr_matrix["target"].drop("target")

# 절대값 기준 정렬
top_features = target_corr.abs().sort_values(ascending=False)

print("=== target 변수와의 상관계수 순위 ===")
for feat in top_features.index:
    orig_val = target_corr[feat]
    print(f"피처: {feat:<12} | 상관계수: {orig_val:>7.4f} (절대값: {abs(orig_val):.4f})")


**결과 해석 / 확인한 점:**  
1. **최강 상관 피처:** `thalach`(최대 심박수, r = +0.669)와 `cp`(흉통 유형, r = +0.573)가 심장 질환과 가장 강한 양의 상관관계를 나타냈습니다.
2. **머신러닝 피처 선택(Feature Selection):** 회귀 및 분류 모델링 구축 시 심박수와 흉통 특성이 가장 중요한 예측 설명변수로 채택되어야 함을 뒷받침합니다.


## 문제 6. 고위험 환자군(심박수 160 이상 & 혈압 130 이상) 필터링 및 리스크 요약 [리스크 스크리닝]

병원 응급실에서 즉각적인 심장 정밀 검사가 필요한 고위험군 조건을 다음과 같이 수립했습니다:
- 조건: `thalach >= 160` (최대 심박수 160 이상) **AND** `trestbps >= 130` (혈압 130 이상)
1. 위 조건을 만족하는 고위험군 환자들을 필터링하여 `high_risk_patients` 데이터프레임을 생성하세요.
2. 이 고위험군 환자들 중 실제 심장 질환 진단을 받은 환자의 비율(발병률)을 구하세요.


In [ ]:
# 문제 6 풀이
# 고위험군 복합 조건 마스킹
condition = (hd_df["thalach"] >= 160) & (hd_df["trestbps"] >= 130)
high_risk_patients = hd_df[condition]

print(f"고위험군 스크리닝 대상 환자 수: {len(high_risk_patients)}명 (총 20명 중)")
print("\n=== 고위험군 환자 명단 ===")
print(high_risk_patients[["patient_id", "age", "trestbps", "thalach", "target"]])

# 발병률 계산
risk_rate = high_risk_patients["target"].mean() * 100
print(f"\n고위험 스크리닝 환자의 실제 질환 발병률: {risk_rate:.1f}%")


**결과 해석 / 확인한 점:**  
1. **스크리닝 룰의 정밀도:** 심박수 160 이상 및 혈압 130 이상 조건을 적용했을 때 총 7명의 환자가 필터링되었으며, 이 중 무려 85.7%(6명)가 실제 심장 질환자로 판정되었습니다.
2. **임상 진단 보조(CDSS):** EDA를 통해 발견한 두 피처의 임계치(Threshold) 조합이 응급 선별 진료에 매우 효과적인 임상 의사결정 규칙이 될 수 있음을 입증했습니다.
